In [1]:
import ast
import os
import jsonlines
from Bio import SeqIO
from Bio.Seq import Seq
from collections import defaultdict

def count_kmer_with_revcomp(dna_seq, k):
    kmer_count = defaultdict(int)
    seq_len = len(dna_seq)
    
    seq = Seq(dna_seq)
    rev_comp_seq = str(seq.reverse_complement())
    
    for i in range(seq_len - k + 1):
        kmer = dna_seq[i:i+k]
        kmer_count[kmer] += 1
    
    for i in range(seq_len - k + 1):
        kmer = rev_comp_seq[i:i+k]
        kmer_count[kmer] += 1
        
    return dict(kmer_count)

os.chdir('/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data')
acc_info = []
with open('assembly_data_report.jsonl', 'r') as file:
    for line in jsonlines.Reader(file):
        acc_info.append(line['accession'])

print(len(acc_info))

51772


In [2]:
from tqdm import tqdm
import multiprocessing

def write_kmer_file(acc_n, que):
    handle = open(f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/data/{acc_n}/genomic.gbff')
    seq_record = SeqIO.parse(handle, 'genbank')
    kmer_dir = f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/kmer/{acc_n}'
    os.makedirs(kmer_dir, exist_ok=True)
    for record in seq_record:
        dna_sequence = str(record.seq)
        kmer_info = {}
        for k in range(5):
            res = count_kmer_with_revcomp(dna_sequence, k+1)
            kmer_info[f'{k+1}-mer'] = res
        file = open(f'{kmer_dir}/{record.id}.txt', 'w+')
        file.write(str(kmer_info))
        file.close()
    que.put(1)


manager = multiprocessing.Manager()
que = manager.Queue()

par = 32
tot = len(acc_info)
pool = multiprocessing.Pool(par)

for acc_n in acc_info:
    pool.apply_async(write_kmer_file, (acc_n, que))
    
pool.close()

count = 0
with tqdm(total = len(acc_info), desc=f'kmer', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
    while True:
        if not que.empty():
            value = que.get(True)
            count += 1
            pbar.update(1)
            if count == tot:
                break
        else:
            continue

IOPub message rate exceeded.██████████████▉                 | 33.7k/51.8k [2:27:15<1:17:36, 3.88B/s]
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

kmer: 100%|███████████████████████████████████████████████████| 51.8k/51.8k [3:53:49<00:00, 3.69B/s]
